# Multi-Class Seasonal Accuracy Comparison: CART vs RF vs KNN vs SVM vs XGBoost

This notebook evaluates all five multi-class classifiers across **3 phenological seasons**
(Early Summer, Post-Monsoon, Winter) to compare performance stability.

> Uses the same training points, bands, and classifier hyperparameters as the
> individual notebooks in `nb/multi/`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import ee
import geemap

# ── Constants ────────────────────────────────────────────────────────────────

CAMPUS_GEOJSON = {
    "type": "Polygon",
    "coordinates": [
        [
            [80.01710357666015, 23.173962177472703],
            [80.03259601593017, 23.165361215115187],
            [80.03654422760009, 23.172502420044232],
            [80.026802444458, 23.181694681000845],
            [80.01542987823485, 23.176960548201308],
        ]
    ],
}

BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI", "NDBI", "SAVI"]
SEED = 42

WATER_POINTS_ASSET = "users/cosypix/water_points"
FOREST_POINTS_ASSET = "users/cosypix/forest_points"
SOIL_POINTS_ASSET = "users/cosypix/soil_points"
BUILDINGS_POINTS_ASSET = "users/cosypix/buildings_points"

CLASSIFIER_CONFIGS = {
    "cart": {"maxNodes": 10, "minLeafPopulation": 10},
    "rf": {"numberOfTrees": 200, "minLeafPopulation": 1, "bagFraction": 0.3, "maxNodes": 10},
    "knn": {"k": 5, "searchMethod": "AUTO", "metric": "EUCLIDEAN"},
    "svm": {"kernelType": "RBF", "gamma": 1.0, "cost": 100.0},
    "xgb": {"numberOfTrees": 200, "shrinkage": 0.05, "maxNodes": 5},
}

CLASSIFIER_DISPLAY_NAMES = {
    "cart": "CART",
    "rf": "Random Forest",
    "knn": "k-NN",
    "svm": "SVM (RBF)",
    "xgb": "XGBoost",
}

def init_ee():
    """Initialize Earth Engine using EE_PROJECT_ID from the .env file."""
    load_dotenv(find_dotenv())
    ee_project = os.getenv("EE_PROJECT_ID")
    if not ee_project:
        raise ValueError("EE_PROJECT_ID not set in .env file")
    ee.Initialize(project=ee_project)
    print("Earth Engine initialized successfully.")

def get_campus_geometry():
    """Return the campus boundary as an ee.Geometry."""
    return ee.Geometry(CAMPUS_GEOJSON)

def mask_s2_clouds(image):
    """Apply Sentinel-2 QA60 cloud/cirrus mask and scale to reflectance."""
    qa = image.select("QA60")
    cloud = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

def build_image(roi, start_date, end_date, cloud_cover=20):
    """Build a cloud-masked Sentinel-2 median composite with spectral indices."""
    dataset = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_cover))
        .map(mask_s2_clouds)
    )
    image = dataset.median().clip(roi)
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndbi = image.normalizedDifference(["B11", "B8"]).rename("NDBI")
    savi = image.expression(
        "1.5 * (NIR - RED) / (NIR + RED + 0.5)",
        {"NIR": image.select("B8"), "RED": image.select("B4")}
    ).rename("SAVI")
    image = image.addBands([ndvi, ndwi, ndbi, savi])
    return image

def load_training_points():
    """Load and merge multi-class training point FeatureCollections."""
    water = ee.FeatureCollection(WATER_POINTS_ASSET)
    forest = ee.FeatureCollection(FOREST_POINTS_ASSET)
    soil = ee.FeatureCollection(SOIL_POINTS_ASSET)
    buildings = ee.FeatureCollection(BUILDINGS_POINTS_ASSET)
    training_points = water.merge(forest).merge(soil).merge(buildings)
    return water, forest, soil, buildings, training_points

def sample_and_split(image, training_points, bands=None, seed=None, split=0.7):
    """Sample *image* at *training_points* and split into train/test sets."""
    if bands is None:
        bands = BANDS
    if seed is None:
        seed = SEED
    training = image.select(bands).sampleRegions(
        collection=training_points,
        properties=["label"],
        scale=10,
    )
    training = training.filter(ee.Filter.notNull(bands + ["label"]))
    training = training.randomColumn("random", seed)
    train_set = training.filter(ee.Filter.lt("random", split))
    test_set = training.filter(ee.Filter.gte("random", split))
    return train_set, test_set

def create_classifier(model_type):
    """Return an **untrained** ee.Classifier with canonical hyperparameters."""
    if model_type not in CLASSIFIER_CONFIGS:
        raise ValueError(f"Unknown model_type '{model_type}'. Choose from {list(CLASSIFIER_CONFIGS.keys())}")
    params = CLASSIFIER_CONFIGS[model_type]
    if model_type == "cart":
        return ee.Classifier.smileCart(**params)
    elif model_type == "rf":
        return ee.Classifier.smileRandomForest(**params)
    elif model_type == "knn":
        return ee.Classifier.smileKNN(**params)
    elif model_type == "svm":
        return ee.Classifier.libsvm(**params)
    elif model_type == "xgb":
        return ee.Classifier.smileGradientTreeBoost(**params)

def get_classifier_factories():
    return {
        CLASSIFIER_DISPLAY_NAMES[key]: (lambda k=key: create_classifier(k))
        for key in CLASSIFIER_CONFIGS
    }


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

init_ee()

### Campus Boundary

In [ ]:
campus = get_campus_geometry()

### Set Bands

In [ ]:
bands = BANDS

### Load Training Points

In [ ]:
water_points, forest_points, soil_points, buildings_points, training_points = load_training_points()

print(f"Water points:     {water_points.size().getInfo()}")
print(f"Forest points:    {forest_points.size().getInfo()}")
print(f"Soil points:      {soil_points.size().getInfo()}")
print(f"Buildings points: {buildings_points.size().getInfo()}")

### Define Seasons & Classifiers

In [ ]:
SEASONS = {
    "Early Summer":  ("2025-03-01", "2025-04-30"),
    "Post-Monsoon":  ("2025-09-01", "2025-10-31"),
    "Winter":         ("2025-11-01", "2025-12-31"),
}

CLOUD_COVER = 20  # default threshold used in multi-class notebooks
CLASSIFIERS = get_classifier_factories()

print(f"Seasons:     {list(SEASONS.keys())}")
print(f"Classifiers: {list(CLASSIFIERS.keys())}")
print(f"Cloud Cover: {CLOUD_COVER}%")

### Run Season × Classifier Sweep

In [ ]:
results = []

for season_name, (start_date, end_date) in SEASONS.items():
    print(f"{'='*60}")
    print(f"  Season: {season_name}  ({start_date} → {end_date})")
    print(f"{'='*60}")

    # Build the median composite for this season
    dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterDate(start_date, end_date)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_COVER))
               .map(mask_s2_clouds))

    img_count = dataset.size().getInfo()
    print(f"  Available images: {img_count}")

    if img_count == 0:
        print(f"  ⚠ No images – skipping")
        for clf_name in CLASSIFIERS:
            results.append({
                'season': season_name, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': 0,
            })
        continue

    image = dataset.median().clip(campus)

    # Add spectral indices
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    savi = image.expression(
        '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
        {'NIR': image.select('B8'), 'RED': image.select('B4')}
    ).rename('SAVI')
    image = image.addBands([ndvi, ndwi, ndbi, savi])

    # Sample and split 
    train_set, test_set = sample_and_split(image, training_points, bands)

    for clf_name, clf_factory in CLASSIFIERS.items():
        try:
            classifier = clf_factory().train(
                features=train_set,
                classProperty='label',
                inputProperties=bands
            )

            validated = test_set.classify(classifier)
            cm = validated.errorMatrix('label', 'classification')
            accuracy = cm.accuracy().getInfo()
            kappa = cm.kappa().getInfo()
            cm_array = cm.getInfo()

            print(f"  {clf_name:25s}  Accuracy: {accuracy:.4f}   Kappa: {kappa:.4f}")
            print(f"  {'':25s}  Confusion Matrix: {cm_array}")

            results.append({
                'season': season_name,
                'classifier': clf_name,
                'accuracy': accuracy,
                'kappa': kappa,
                'confusion_matrix': str(cm_array),
                'image_count': img_count,
            })
        except Exception as e:
            print(f"  {clf_name:25s}  ERROR: {e}")
            results.append({
                'season': season_name, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': img_count,
            })

print("✅ Sweep complete!")

### Results Table

In [ ]:
df = pd.DataFrame(results)
print(df[['season', 'classifier', 'accuracy', 'kappa']].to_string(index=False))

### Accuracy Heatmap (Season × Classifier)

In [ ]:
# Pivot for heatmap
pivot_acc = df.pivot(index='classifier', columns='season', values='accuracy')
pivot_kap = df.pivot(index='classifier', columns='season', values='kappa')

# Reorder columns to match presentation
season_order = ['Early Summer', 'Post-Monsoon', 'Winter']
pivot_acc = pivot_acc[[s for s in season_order if s in pivot_acc.columns]]
pivot_kap = pivot_kap[[s for s in season_order if s in pivot_kap.columns]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy heatmap
sns.heatmap(pivot_acc, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=1, linecolor='white', ax=axes[0],
            vmin=0.5, vmax=1.0)
axes[0].set_title('Multi-Class Model Stability Across Seasons (Accuracy)', fontweight='bold', pad=12)
axes[0].set_ylabel('')
axes[0].set_xlabel('')

# Kappa heatmap
sns.heatmap(pivot_kap, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=1, linecolor='white', ax=axes[1],
            vmin=0.0, vmax=1.0)
axes[1].set_title('Multi-Class Kappa Statistic Across Seasons', fontweight='bold', pad=12)
axes[1].set_ylabel('')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('multi_seasonal_accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/multi_seasonal_accuracy_heatmap.png")

### Bar Chart Comparison

In [ ]:
colors = {
    "CART": "#9b59b6",
    "Random Forest": "#2ecc71",
    "k-NN": "#f39c12",
    "SVM (RBF)": "#e74c3c",
    "XGBoost": "#3498db",
}

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(season_order))
width = 0.15

for i, clf_name in enumerate(CLASSIFIERS):
    subset = df[df['classifier'] == clf_name].set_index('season')
    vals = [subset.loc[s, 'accuracy'] if s in subset.index and pd.notna(subset.loc[s, 'accuracy']) else 0
            for s in season_order]
    ax.bar(x + i * width, vals, width, label=clf_name, color=colors.get(clf_name, '#95a5a6'))

ax.set_xticks(x + width * 2)
ax.set_xticklabels(season_order)
ax.set_ylabel('Accuracy')
ax.set_title('Multi-Class Classification Accuracy by Season and Classifier', fontweight='bold')
ax.legend()
ax.set_ylim(0.4, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('multi_seasonal_accuracy_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/multi_seasonal_accuracy_bars.png")

### Save Results

In [ ]:
df.to_csv('multi_seasonal_accuracy_results.csv', index=False)
print("Saved: fe/multi_seasonal_accuracy_results.csv")

# Summary
print("" + "="*60)
print("  BEST ACCURACY PER SEASON")
print("="*60)
for season in season_order:
    subset = df[(df['season'] == season) & df['accuracy'].notna()]
    if not subset.empty:
        best = subset.loc[subset['accuracy'].idxmax()]
        print(f"  {season:20s}  {best['classifier']:20s}  Accuracy: {best['accuracy']:.4f}")